### NLP Assignment 1 - Sentence Generator using N-gram
Team Members: ID1_ID2_ID3_ID4

In [1]:
# ============================================
# CELL 1: Imports and Common Interface
# ============================================

import nltk
from nltk.corpus import brown
import random
import string
from collections import defaultdict

# Download required NLTK data
nltk.download('brown')
nltk.download('punkt')


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

COMMON INTERFACE DEFINITIONS
DO NOT MODIFY

DATA STRUCTURES AGREEMENT:
1. preprocessed_corpus: list of lists
   Each inner list is a sentence, each element is a word (string)
   Example: [["the", "cat", "sat"], ["dogs", "bark"]]

2. bigram_dict: dictionary with tuple keys and int values
   Key: (word1, word2) tuple of two words
   Value: frequency count
   Example: {("the", "cat"): 5, ("cat", "sat"): 3}

3. trigram_dict: dictionary with tuple keys and int values
   Key: (word1, word2, word3) tuple of three words
   Value: frequency count
   Example: {("the", "cat", "sat"): 2, ("cat", "sat", "on"): 1}

In [2]:
# ============================================
# CELL 2: Person 1 - UI and Input Handling (Ahmed - 20220040)
# ============================================

def get_user_input():
    """
    Person 1's function
    Returns: tuple (M, N, maxLen) as integers
    """
    while True:
        try:
            M = int(input("Enter number of sentences to generate (M): "))
            if M <= 0:
                print("M must be a positive integer. Please try again.")
                continue
            
            N = int(input("Enter n-gram order (2 for bigram, 3 for trigram): "))
            if N not in [2, 3]:
                print("N must be 2 or 3. Please try again.")
                continue
            
            maxLen = int(input("Enter maximum sentence length (maxLen): "))
            if maxLen <= 0:
                print("maxLen must be a positive integer. Please try again.")
                continue
            
            return M, N, maxLen
        
        except ValueError:
            print("Invalid input. Please enter integers only.")

def display_sentences(sentences):
    """
    Person 1's function
    Input: list of generated sentences (each sentence is a list of words)
    Output: prints sentences nicely
    """
    for i, sentence in enumerate(sentences, 1):
        # Join words into a single string
        sentence_str = ' '.join(sentence)
        print(f"Sentence {i}: {sentence_str}")

In [3]:
# ============================================
# CELL 3: Person 2 - Data Collection and Preprocessing (Sobhi - 20221088)
# ============================================

def remove_punctuation(word):
    """
    Helper function for Person 2
    Input: word string
    Returns: word string with punctuation removed
    """
    # Remove punctuation from the word
    return ''.join(char for char in word if char not in string.punctuation)

def load_and_preprocess_corpus():
    """
    Person 2's function
    Returns: preprocessed_corpus (list of sentences, each sentence is list of words)
    Requirements:
    1. Get sentences from brown corpus until reaching 200,000 tokens (±100)
    2. Tokenize each sentence into words
    3. Remove punctuation from tokens
    Note: 200,000 is limit on original corpus tokens, not vocabulary
    """
    preprocessed_corpus = []
    total_tokens = 0
    target_tokens = 200000
    tolerance = 100
    
    print("Loading Brown corpus...")
    
    # Get all sentences from brown corpus
    for sentence in brown.sents():
        if total_tokens >= target_tokens + tolerance:
            break
            
        # Process each sentence
        processed_sentence = []
        for word in sentence:
            if total_tokens >= target_tokens + tolerance:
                break
                
            # Remove punctuation and convert to lowercase
            clean_word = remove_punctuation(word.lower())
            
            # Only add non-empty words
            if clean_word:
                processed_sentence.append(clean_word)
                total_tokens += 1
        
        # Add sentence to corpus if it has at least one word
        if processed_sentence:
            preprocessed_corpus.append(processed_sentence)
    
    print(f"✓ Total tokens collected: {total_tokens}")
    print(f"✓ Total sentences collected: {len(preprocessed_corpus)}")
    
    return preprocessed_corpus

def get_vocabulary(preprocessed_corpus):
    """
    Create vocabulary set from corpus tokens
    Returns: set of unique words in the corpus
    """
    vocabulary = set()
    for sentence in preprocessed_corpus:
        vocabulary.update(sentence)
    print(f"✓ Vocabulary size: {len(vocabulary)} unique words")
    return vocabulary

In [4]:
# ============================================
# CELL 4: Person 3 - N-gram Model Building (Fixed)
# ============================================
def build_bigram_model(preprocessed_corpus):
    bigram_dict = defaultdict(int)
    for sentence in preprocessed_corpus:
        for i in range(len(sentence) - 1):
            bigram = (sentence[i], sentence[i+1])
            bigram_dict[bigram] += 1
    return dict(bigram_dict)

def build_trigram_model(preprocessed_corpus):
    trigram_dict = defaultdict(int)
    for sentence in preprocessed_corpus:
        for i in range(len(sentence) - 2):
            trigram = (sentence[i], sentence[i+1], sentence[i+2])
            trigram_dict[trigram] += 1
    return dict(trigram_dict)

def get_ngram_counts(preprocessed_corpus, n):
    ngram_dict = defaultdict(int)
    for sentence in preprocessed_corpus:
        for i in range(len(sentence) - n + 1):
            ngram = tuple(sentence[i:i+n])
            ngram_dict[ngram] += 1
    return dict(ngram_dict)

In [5]:
# ============================================
# CELL 5: Person 4 - Sentence Generator and Integration (Mariam - 20221217)
# ============================================

def calculate_bigram_probability(word, prev_word, bigram_dict):
    """
    Helper function for Person 4
    Returns: P(word | prev_word) = count(prev_word, word) / count(prev_word)
    """
    # Count all bigrams starting with prev_word
    total_count = 0
    for (w1, w2), count in bigram_dict.items():
        if w1 == prev_word:
            total_count += count
    
    if total_count == 0:
        return 0
    
    # Get count for specific bigram
    bigram_count = bigram_dict.get((prev_word, word), 0)
    
    return bigram_count / total_count

def calculate_trigram_probability(word, prev_word1, prev_word2, trigram_dict):
    """
    Helper function for Person 4
    Returns: P(word | prev_word1, prev_word2) = count(prev_word1, prev_word2, word) / count(prev_word1, prev_word2)
    """
    # Count all trigrams starting with (prev_word1, prev_word2)
    total_count = 0
    for (w1, w2, w3), count in trigram_dict.items():
        if w1 == prev_word1 and w2 == prev_word2:
            total_count += count
    
    if total_count == 0:
        return 0
    
    # Get count for specific trigram
    trigram_count = trigram_dict.get((prev_word1, prev_word2, word), 0)
    
    return trigram_count / total_count

def get_random_starter(n, bigram_dict, trigram_dict, preprocessed_corpus):
    """
    Helper function for Person 4
    Returns: random (n-1)gram as tuple to start sentence generation
    """
    if n == 2:
        # For bigram, return random single word
        all_words = []
        for sentence in preprocessed_corpus:
            all_words.extend(sentence)
        if all_words:
            return (random.choice(all_words),)
        return None
    
    elif n == 3:
        # For trigram, return random word pair from bigrams
        if bigram_dict:
            random_bigram = random.choice(list(bigram_dict.keys()))
            return random_bigram
        return None

def select_next_word_top3(current_context, n, bigram_dict, trigram_dict, vocabulary):
    """
    Updated helper function for Person 4
    Selects from top 3 words with highest probability (randomly)
    Input: current_context (tuple of (n-1) words), n, and appropriate dictionaries
    Returns: next word selected randomly from top 3 probabilities
    """
    word_probabilities = []
    
    if n == 2:
        prev_word = current_context[0]
        for word in vocabulary:
            prob = calculate_bigram_probability(word, prev_word, bigram_dict)
            if prob > 0:
                word_probabilities.append((word, prob))
    
    elif n == 3:
        prev_word1, prev_word2 = current_context
        for word in vocabulary:
            prob = calculate_trigram_probability(word, prev_word1, prev_word2, trigram_dict)
            if prob > 0:
                word_probabilities.append((word, prob))
    
    # Sort by probability in descending order
    word_probabilities.sort(key=lambda x: x[1], reverse=True)
    
    # Take top 3 (or fewer if less than 3 words have probability > 0)
    top_words = word_probabilities[:3]
    
    if not top_words:
        # If no words found with probability > 0, return random word from vocabulary
        return random.choice(list(vocabulary))
    
    # Select randomly from top 3
    selected_word = random.choice(top_words)[0]
    return selected_word

def generate_sentences(M, N, maxLen, bigram_dict, trigram_dict, preprocessed_corpus, vocabulary):
    """
    Person 4's main function - Updated with new specification
    Input: user parameters, models, and vocabulary
    Returns: list of generated sentences (each sentence is list of words)
    """
    generated_sentences = []
    
    for sentence_num in range(M):
        # Get random starter
        starter = get_random_starter(N, bigram_dict, trigram_dict, preprocessed_corpus)
        
        if starter is None:
            # Fallback: start with random word from vocabulary
            starter = (random.choice(list(vocabulary)),)
        
        # Initialize sentence with starter
        if N == 2:
            sentence = [starter[0]]
            current_context = starter
        else:  # N == 3
            sentence = [starter[0], starter[1]]
            current_context = starter
        
        # Generate remaining words
        while len(sentence) < maxLen:
            # Select next word from top 3 probabilities
            next_word = select_next_word_top3(current_context, N, bigram_dict, trigram_dict, vocabulary)
            
            if next_word is None:
                # Fallback to random word if selection fails
                next_word = random.choice(list(vocabulary))
            
            sentence.append(next_word)
            
            # Update context for next iteration
            if N == 2:
                current_context = (next_word,)
            else:  # N == 3
                current_context = (current_context[1], next_word)
        
        generated_sentences.append(sentence)
        
        # Optional: Show progress
        if (sentence_num + 1) % 10 == 0:
            print(f"  Generated {sentence_num + 1}/{M} sentences...")
    
    return generated_sentences

In [6]:
# ============================================
# CELL 6: Person 4 - Main Integration Function (Updated)
# ============================================

def main():
    """
    Person 4 - Integration of all components
    This function calls all team members' functions in order
    """
    print("=" * 50)
    print("NLP Assignment 1 - Sentence Generator")
    print("Team: 20221217_20220136_20221088_20220040")
    print("=" * 50)
    
    try:
        # Person 1's part
        print("\n[Step 1] Getting user input...")
        M, N, maxLen = get_user_input()
        print(f"✓ Generating {M} sentences with N={N}, maxLen={maxLen}")
        
        # Person 2's part
        print("\n[Step 2] Loading and preprocessing Brown corpus...")
        corpus = load_and_preprocess_corpus()
        total_tokens = sum(len(sent) for sent in corpus)
        print(f"✓ Corpus loaded with {total_tokens} tokens in {len(corpus)} sentences")
        
        # Get vocabulary from corpus
        vocabulary = get_vocabulary(corpus)
        
        # Person 3's part
        print("\n[Step 3] Building n-gram models...")
        bigram_dict = build_bigram_model(corpus)
        trigram_dict = build_trigram_model(corpus)
        print(f"✓ Bigram model built with {len(bigram_dict)} unique bigrams")
        print(f"✓ Trigram model built with {len(trigram_dict)} unique trigrams")
        
        # Person 4's part
        print(f"\n[Step 4] Generating {M} sentences (selecting from top 3 probabilities)...")
        generated_sentences = generate_sentences(M, N, maxLen, bigram_dict, trigram_dict, corpus, vocabulary)
        print(f"✓ Generated {len(generated_sentences)} sentences")
        
        # Person 1's part again
        print("\n[Step 5] Displaying generated sentences:")
        print("-" * 40)
        display_sentences(generated_sentences)
        print("-" * 40)
        
        print("\n" + "=" * 50)
        print("✓ Assignment completed successfully!")
        print("=" * 50)
        
    except Exception as e:
        print(f"\n❌ An error occurred: {e}")
        import traceback
        traceback.print_exc()
        print("Please check your input and try again.")

In [9]:
# ============================================
# CELL 7: Run the Program
# ============================================

# Execute the main function when notebook runs
if __name__ == "__main__":
    main()

NLP Assignment 1 - Sentence Generator
Team: 20221217_20220136_20221088_20220040

[Step 1] Getting user input...
✓ Generating 6 sentences with N=3, maxLen=20

[Step 2] Loading and preprocessing Brown corpus...
Loading Brown corpus...
✓ Total tokens collected: 200100
✓ Total sentences collected: 10334
✓ Corpus loaded with 200100 tokens in 10334 sentences
✓ Vocabulary size: 21009 unique words

[Step 3] Building n-gram models...
✓ Bigram model built with 112815 unique bigrams
✓ Trigram model built with 162712 unique trigrams

[Step 4] Generating 6 sentences (selecting from top 3 probabilities)...


KeyboardInterrupt: 

In [8]:
# ============================================
# CELL 8: Test Cases (Optional)
# ============================================

def test_with_small_corpus():
    """Test the entire pipeline with a small corpus"""
    print("=" * 40)
    print("Testing with small corpus...")
    print("=" * 40)
    
    # Create small test corpus
    test_corpus = [
        ["the", "cat", "sat", "on", "the", "mat"],
        ["the", "dog", "ran", "in", "the", "park"],
        ["the", "cat", "chased", "the", "mouse"],
        ["dogs", "bark", "loudly", "at", "night"],
        ["cats", "and", "dogs", "are", "pets"]
    ]
    
    print(f"Test corpus: {len(test_corpus)} sentences")
    
    # Get vocabulary
    vocabulary = set()
    for sent in test_corpus:
        vocabulary.update(sent)
    print(f"Vocabulary size: {len(vocabulary)} words")
    
    # Build models
    bigram_dict = build_bigram_model(test_corpus)
    trigram_dict = build_trigram_model(test_corpus)
    
    print(f"Bigram dict: {len(bigram_dict)} entries")
    print(f"Trigram dict: {len(trigram_dict)} entries")
    
    # Test probability calculations
    print("\n--- Testing Probability Calculations ---")
    prob = calculate_bigram_probability("cat", "the", bigram_dict)
    print(f"P(cat | the) = {prob:.3f}")
    
    prob = calculate_trigram_probability("sat", "the", "cat", trigram_dict)
    print(f"P(sat | the, cat) = {prob:.3f}")
    
    # Test top 3 selection
    print("\n--- Testing Top 3 Selection ---")
    context = ("the",)
    print(f"Context: {context}")
    word_probs = []
    for word in vocabulary:
        prob = calculate_bigram_probability(word, "the", bigram_dict)
        if prob > 0:
            word_probs.append((word, prob))
    
    word_probs.sort(key=lambda x: x[1], reverse=True)
    print("Top 3 words after 'the':")
    for i, (word, prob) in enumerate(word_probs[:3], 1):
        print(f"  {i}. '{word}' (prob: {prob:.3f})")
    
    # Test sentence generation with new top-3 selection
    print("\n--- Testing Sentence Generation (Top 3 Selection) ---")
    sentences = generate_sentences(5, 2, 6, bigram_dict, trigram_dict, test_corpus, vocabulary)
    display_sentences(sentences)
    
    print("\n✓ All tests passed!")
    return True

# Uncomment to run tests
test_with_small_corpus()

Testing with small corpus...
Test corpus: 5 sentences
Vocabulary size: 20 words
Bigram dict: 21 entries
Trigram dict: 17 entries

--- Testing Probability Calculations ---
P(cat | the) = 0.333
P(sat | the, cat) = 0.500

--- Testing Top 3 Selection ---
Context: ('the',)
Top 3 words after 'the':
  1. 'cat' (prob: 0.333)
  2. 'park' (prob: 0.167)
  3. 'dog' (prob: 0.167)

--- Testing Sentence Generation (Top 3 Selection) ---
Sentence 1: chased the cat chased the cat
Sentence 2: cat chased the cat sat on
Sentence 3: loudly at night sat on the
Sentence 4: the cat chased the cat sat
Sentence 5: ran in the park are pets

✓ All tests passed!


True